# 第 19 课｜KV cache、Prefill 与 Decode

**主线问题：不改变模型参数和预测含义，怎样避免生成时反复重算前缀？**

本课交付：独立 Jupyter 讲义与同名完整 `.py` 脚本。不需要前课源码、不联网、不下载大模型。
有第 17/18 课检查点时可直接加载；没有时训练一个小型模板模型。

先完成第 16–18 课，能解释 `[B,H,T,d]`、因果 mask、绝对位置向量和逐 token 生成。
建议分两次学习：先写出缓存前向并验证一致性，再测量速度、内存并完成练习。

> 本课缓存路径仅支持**相同长度、没有 padding 的批次**，默认 FP32。
> 普通全量 `forward` 仍支持前课的右侧 padding 训练。缓存不是长期记忆，也不会扩大位置表。
> GPU 不是必需品；本文件的保存输出来自 CPU。不要据此推算 RTX 3060 的实际速度。

## 0. 学完后能做什么？

| 能力 | 验收方式 |
|---|---|
| 解释 Prefill 与 Decode | 说清第一枚新 token 是在哪次前向后选出的 |
| 缓存每层 K/V | 能写出形状、追加轴与位置偏移 |
| 处理矩形因果 mask | `past_len=3, new_len=2` 时手写正确的 2×5 矩阵 |
| 验证等价性 | 整段、逐 token、分块输入的每个位置 logits 接近 |
| 测量而不猜性能 | 固定权重与 token 路径，分开记录 prefill / decode |
| 估算内存 | 从 K/V 形状推导有效元素载荷，说明它不等于峰值显存 |

阅读次序：0–6 节原理和实现 → 7–10 节实验 → 11 节独立练习 → 12 节答案。

## 1. 环境与配置

沿用已有环境，无需为了本课重装 PyTorch。若没有安装依赖，在终端执行：

```bash
python -m pip install torch jupyterlab ipykernel
```

GPU 构建请按 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/) 选择。
Notebook 使用哪个内核，就把包安装到哪个 Python 环境。

`DEVICE='cpu'` 是默认值；已有可用 CUDA 环境可以改成 `'auto'`。
`CHECKPOINT=None` 会训练演示模型；改为你自己的检查点路径就跳过训练。
只加载自己生成或信任的文件；`weights_only=True` 不是任意文件安全性的保证。

下面复用配置类和基础工具。不会在导入时训练、创建数据或改全局随机状态。

In [1]:
from __future__ import annotations
import argparse
from dataclasses import asdict, dataclass
from functools import partial
import hashlib
import json
import math
from pathlib import Path
import random
import tempfile
import statistics
import time
import platform
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100

def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")


def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)


def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class GPTConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.1

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ff_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= len(SPECIAL_TOKENS):
            raise ValueError("词表除了 4 个特殊 token，还必须有普通字符")
        if self.d_model % self.n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")

DEVICE = "cpu"
CHECKPOINT = None  # 示例：Path("lesson_17_outputs/best_model.pt")
OUTPUT_DIR = Path("lesson_19_outputs")
EPOCHS = 40       # 只有 CHECKPOINT=None 时使用
RUN_BENCHMARK = True
BENCHMARK_REPEATS = 7

torch.set_num_threads(1)  # 微型 CPU 实验使用单线程，避免线程调度掩盖比较
DEVICE = choose_device(DEVICE)
print("PyTorch:", torch.__version__)
print("设备:", DEVICE, "CUDA可用:", torch.cuda.is_available())
print("输出目录:", OUTPUT_DIR.resolve())

PyTorch: 2.10.0+cpu
设备: cpu CUDA可用: False
输出目录: /mnt/data/_lesson19_work/notebook_run/lesson_19_outputs


## 2. 为什么先处理整段，再逐 token 处理？

假设提示有 $P=6$ 个 token，我们要产生 $N=4$ 个新 token。

**没有 cache**：每步把整个当前前缀重新送进模型，依次处理 6、7、8、9 个位置。

**有 cache**：先处理 6 个提示位置并缓存所有层的 K/V，这叫 **Prefill**。
它的最后一个位置 logits 已能选出第一枚新 token。随后把刚选出的 token 送回模型，
连同历史 K/V 计算下一个预测，这个增量阶段叫 **Decode**。

因此，4 次预测需要 **1 次 prefill + 3 次 decode**，不是 1+4 次。

| 调用 | 本次送入缓存版的 token | 调用后 cache 长度 | 调用的 logits 用来选 |
|---|---|---:|---|
| Prefill | 提示的 6 个 token | 6 | 第 1 枚新 token |
| Decode 1 | 第 1 枚新 token | 7 | 第 2 枚新 token |
| Decode 2 | 第 2 枚新 token | 8 | 第 3 枚新 token |
| Decode 3 | 第 3 枚新 token | 9 | 第 4 枚新 token |

**刚被选出的 token，还没有经过下一次前向，就没有自己的 K/V。**
最后的文本有 10 个 token，cache 可以只有 9 个。这不是自动少缓存了一个位置。

参考：[Hugging Face — How caching works](https://huggingface.co/docs/transformers/cache_explanation)。

In [2]:
P, N = 6, 4
full_lengths = [P + step for step in range(N)]
cached_lengths = [P] + [1] * (N - 1)
print("无缓存每次处理:", full_lengths, "合计:", sum(full_lengths))
print("有缓存每次处理:", cached_lengths, "合计:", sum(cached_lengths))
print("输出总长度:", P + N, "最后一次前向已处理长度:", P + N - 1)

无缓存每次处理: [6, 7, 8, 9] 合计: 30
有缓存每次处理: [6, 1, 1, 1] 合计: 9
输出总长度: 10 最后一次前向已处理长度: 9


### 2.1 为什么旧 K/V 可以复用？为什么不保存旧 Q？

固定参数、位置编号和前缀，在关闭 Dropout 的确定性因果模型中：
第一层旧位置只能读取自己及更早位置，不受后面新 token 影响；逐位置的 LayerNorm、FFN 也不会混入未来。
所以旧位置在第一层的隐藏表示不变；相同推理逐层成立，旧位置在各层得到的 K/V 也不变。

新增位置仍需要自己的 Q，用它读取所有允许的 K/V。旧位置的输出不需要重新产生，
所以本课缓存不保存旧 Q，也不保存注意力权重矩阵。

这不是“所有 Transformer 都能这样缓存”：若修改早期 token、权重、位置规则，或者允许双向注意力，
上述条件就可能不成立。训练时我们继续整段前向与反向传播，不复用此推理缓存。

参考：[缓存解释与逐层 K/V](https://huggingface.co/docs/transformers/cache_explanation)。

### 2.2 缓存结构和注意力形状

本课约定 `cache[layer] = (K, V)`。每层单独保存一对张量：

| 对象 | 形状 |
|---|---|
| 历史 K / V | `[B,H,P,d]` |
| 本次 Q / K_new / V_new | `[B,H,Q,d]` |
| 追加后的 K / V | `[B,H,P+Q,d]` |
| 新 query 对全部 key 的分数 | `[B,H,Q,P+Q]` |
| 本次注意力输出 | `[B,Q,D]` |

注意 $Q$ 在表中表示**本次新增长度**，不是投影张量本身；$D=H d$。
当 `Q=1` 时，分数不再是 `[T,T]`，而是一行对全部历史的分数。

实现使用 Python tuple 和 `torch.cat(..., dim=-2)`；这只是透明易读的动态缓存，
不是 Hugging Face `Cache` 类，也不是生产级高效缓存分配器。

## 3. 缓存中最关键的两件事：位置偏移和矩形 mask

历史已有 $P$ 个位置，本次新增 $Q$ 个位置。新增位置编号必须是：

$$P,\,P+1,\ldots,P+Q-1$$

不能每次都用 `[0]`，也不能把已缓存前缀重复送进去。

新增 query 的本地索引为 $i$，全部 key 的全局索引为 $j$，允许条件是：

$$j\le P+i$$

下面用**绝对位置比较**构造 mask，避免只凭矩阵形状猜三角线在哪里。
`True` 表示允许读取。

In [3]:
LayerKV = tuple[torch.Tensor, torch.Tensor]
KVCache = tuple[LayerKV, ...]

def offset_causal_mask(past_len: int, new_len: int, device: torch.device | str = 'cpu') -> torch.Tensor:
    """返回 [1,1,Q,P+Q]。第 i 个新 query 的绝对位置是 P+i。True=允许。"""
    if type(past_len) is not int or past_len < 0:
        raise ValueError('past_len 必须是非负整数')
    positive_int(new_len, 'new_len')
    query_positions = past_len + torch.arange(new_len, device=device)
    key_positions = torch.arange(past_len + new_len, device=device)
    allowed = key_positions[None, :] <= query_positions[:, None]
    return allowed[None, None, :, :]

In [4]:
for past, new in [(0, 4), (3, 1), (3, 2)]:
    allowed = offset_causal_mask(past, new)
    print(f"past={past}, new={new}, shape={tuple(allowed.shape)}")
    print(allowed[0, 0].to(torch.int64))

past=0, new=4, shape=(1, 1, 4, 4)
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]])
past=3, new=1, shape=(1, 1, 1, 4)
tensor([[1, 1, 1, 1]])
past=3, new=2, shape=(1, 1, 2, 5)
tensor([[1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]])


### 3.1 一个非常容易误用的接口

`F.scaled_dot_product_attention(..., is_causal=True)` 在非方阵时使用**左上角对齐**的因果规则。
对形状为 `[1,4]` 的分数，它只保留第一个 key；而本课 3 个历史位置加 1 个新位置的 query，
应该能读取全部 4 个 key。

设 Q/K 都为零，V 分别为 1、2、3、4，那么正确输出应为平均值 **2.5**。
盲目传 `is_causal=True` 会得到 **1.0**。

本课在带历史的场景中显式传入偏移 mask，并设 `is_causal=False`。
单 token、无 padding、所有 key 都是历史或当前时，也可以不传 mask；
但追加多个 token 的 chunk 时，仍然需要防止 chunk 内部读取未来。

参考：[PyTorch SDPA 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)。

In [5]:
def sdpa_alignment_demo(device: torch.device = torch.device('cpu')) -> dict:
    """P=3,Q=1。均匀分数，V=[1,2,3,4]，正确答案是均值2.5，不是1。"""
    q = torch.zeros(1, 1, 1, 2, device=device)
    k = torch.zeros(1, 1, 4, 2, device=device)
    v = torch.arange(1, 5, dtype=torch.float32, device=device).reshape(1, 1, 4, 1)
    wrong = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
    right = F.scaled_dot_product_attention(q, k, v, attn_mask=offset_causal_mask(3, 1, device),
                                           dropout_p=0.0, is_causal=False)
    return {'upper_left_is_causal': wrong.item(), 'offset_mask': right.item(), 'expected': 2.5}

print(sdpa_alignment_demo(DEVICE))

{'upper_left_is_causal': 1.0, 'offset_mask': 2.5, 'expected': 2.5}


## 4. 保留原模型，给相同参数增加一条缓存路径

下面复用第 17 课的配置、注意力、Block 和 GPT。已有的 `forward()` 原样保留作为参照。
这一大段不是本课的新知识，可以折叠；但保留在讲义里，保证文件独立运行。

我们稍后定义 `CachedTinyGPT(TinyGPT)`：继承原结构，只增加 `forward_cached()` 方法。
它不创建额外的可训练参数，旧 `state_dict` 键名保持兼容。

In [6]:
def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attention_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        q = self.q_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~allowed, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        heads = self.attention_dropout(weights) @ v
        merged = heads.transpose(1, 2).contiguous().reshape(b, t, d)
        return self.out_proj(merged)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(nn.Linear(cfg.d_model, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.d_model))
        self.residual_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        x = x + self.residual_dropout(self.attn(self.norm1(x), allowed))
        x = x + self.residual_dropout(self.ffn(self.norm2(x)))
        return x


class TinyGPT(nn.Module):
    """输入 [B,T]，输出 [B,T,V] 原始 logits。forward 不计算损失，不移位标签。"""
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.d_model)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        # 每次创建新的 Block；不要使用 [一个Block] * n_layers。
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        # 本课不共享输出头与输入 Embedding 的权重，方便辨认两个角色。
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        if (not isinstance(input_ids, torch.Tensor) or input_ids.dtype != torch.long
                or input_ids.ndim != 2 or min(input_ids.shape) <= 0):
            raise ValueError("input_ids 必须是非空 [B,T] torch.long 张量")
        if input_ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须位于同一设备")
        if bool(((input_ids < 0) | (input_ids >= self.cfg.vocab_size)).any()):
            raise ValueError("输入包含词表范围之外的编号")
        t = input_ids.shape[1]
        if t > self.cfg.max_len:
            raise ValueError(f"输入长度 {t} 超过上下文 {self.cfg.max_len}；本课不自动截断")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        for block in self.blocks:
            x = block(x, allowed)
        return self.lm_head(self.final_norm(x))


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def parameter_formula(cfg: GPTConfig) -> int:
    d, f = cfg.d_model, cfg.ff_dim
    # 每块：4 个 D→D + 两个 LayerNorm + D→F→D。
    per_block = 4 * (d*d + d) + 4*d + (d*f + f) + (f*d + d)
    return 2 * cfg.vocab_size*d + cfg.max_len*d + cfg.n_layers*per_block + 2*d

### 4.1 新增注意力核心

与全量注意力相比，只增加三件事：

1. 只从本次新增的隐藏状态算 Q、K_new、V_new。
2. 沿序列轴追加历史 K/V，并使用偏移后的因果 mask。
3. 返回本次输出，以及扩展后的 K/V。

这里不原地修改旧 cache，因此一个历史可以分出两个后续分支。
但若你手动改写缓存张量，或一直保留所有历史版本，仍会造成错误或额外内存占用。

In [7]:
def cached_attention(attn: CausalSelfAttention, x: torch.Tensor,
                     past: LayerKV | None = None) -> tuple[torch.Tensor, LayerKV]:
    """本函数是内部核心；cache 的层数/形状/设备由模型入口统一校验。

    x 只包含这次新增的位置，不能把已缓存的旧前缀再次传入。
    torch.cat 创建新张量，不原地修改调用者持有的旧 cache。
    """
    b, q_len, d_model = x.shape
    h, dh = attn.n_heads, attn.d_head
    q = attn.q_proj(x).reshape(b, q_len, h, dh).transpose(1, 2)
    k_new = attn.k_proj(x).reshape(b, q_len, h, dh).transpose(1, 2)
    v_new = attn.v_proj(x).reshape(b, q_len, h, dh).transpose(1, 2)
    past_len = 0 if past is None else past[0].shape[-2]
    if past is None:
        k_all, v_all = k_new, v_new
    else:
        k_all = torch.cat((past[0], k_new), dim=-2)
        v_all = torch.cat((past[1], v_new), dim=-2)
    allowed = offset_causal_mask(past_len, q_len, x.device)
    scores = q @ k_all.transpose(-2, -1) / math.sqrt(dh)
    weights = torch.softmax(scores.masked_fill(~allowed, float('-inf')), dim=-1)
    heads = attn.attention_dropout(weights) @ v_all
    merged = heads.transpose(1, 2).contiguous().reshape(b, q_len, d_model)
    return attn.out_proj(merged), (k_all, v_all)

### 4.2 跨层增量前向

`forward_cached(input_ids, past_key_values=None)` 的 `input_ids` **只包含未处理的新 token**。
它返回 `[B,Q,V]` logits 和每层扩展后的 K/V。

每层做法仍是：Pre-LN → Attention → 残差 → Pre-LN → FFN → 残差。
**两层都必须保留自己的 cache，不能只缓存第一层，也不能把第一层的 K/V 套用到第二层。**

本实现的入口额外检查：eval/no_grad、输入 long 类型、无 PAD、cache 层数/形状/长度一致、
设备和 dtype 一致，以及 `past_len+new_len` 不超过位置表。它不自动判断缓存的语义来源。
不同权重或前缀的缓存，即使形状相同，也不能直接复用。

In [8]:
class CachedTinyGPT(TinyGPT):
    """继承已学过的模型，新增方法，不新增可训练参数，不改变 state_dict 键名。"""

    def _validate_cached_inputs(self, input_ids: torch.Tensor,
                                past_key_values: KVCache | None) -> int:
        if any(module.training for module in self.modules()):
            raise ValueError('缓存路径只用于评估；先调用 model.eval()')
        if torch.is_grad_enabled():
            raise ValueError('缓存路径只用于推理；请放在 torch.no_grad() 中')
        device = self.token_embedding.weight.device
        if torch.is_autocast_enabled(device.type):
            raise ValueError('本课缓存路径不支持 autocast，请使用模型自身的统一 dtype')
        if (not isinstance(input_ids, torch.Tensor) or input_ids.dtype != torch.long
                or input_ids.ndim != 2 or min(input_ids.shape) <= 0):
            raise ValueError('input_ids 必须是非空 [B,Q] long 张量')
        if input_ids.device != device:
            raise ValueError('input_ids 与模型设备不一致')
        if bool(((input_ids < 0) | (input_ids >= self.cfg.vocab_size)).any()):
            raise ValueError('token 编号越界')
        if bool((input_ids == PAD_ID).any()):
            raise ValueError('本课缓存路径不支持 PAD；请逐条处理不同长度的提示')
        past_len = 0
        if past_key_values is not None:
            if (not isinstance(past_key_values, tuple)
                    or len(past_key_values) != self.cfg.n_layers):
                raise ValueError('cache 必须是与层数相同的 tuple；空历史请用 None')
            lengths = []
            for layer, pair in enumerate(past_key_values):
                if not isinstance(pair, tuple) or len(pair) != 2:
                    raise ValueError(f'cache 第 {layer} 层必须是 (K,V)')
                k, v = pair
                if not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor):
                    raise ValueError('K/V 必须是张量')
                if (k.ndim != 4 or v.shape != k.shape or k.shape[0] != input_ids.shape[0]
                        or k.shape[1] != self.cfg.n_heads
                        or k.shape[3] != self.cfg.d_model // self.cfg.n_heads or k.shape[2] <= 0):
                    raise ValueError(f'cache 第 {layer} 层需要 [B,H,P,d_head]，且 P>0')
                for tensor in (k, v):
                    if tensor.device != device or tensor.dtype != self.token_embedding.weight.dtype:
                        raise ValueError('cache dtype/device 必须与模型一致；移动模型后需重建')
                    if tensor.requires_grad:
                        raise ValueError('推理 cache 不应保留求导图')
                lengths.append(k.shape[2])
            if len(set(lengths)) != 1:
                raise ValueError('各层 cache 历史长度必须一致')
            past_len = lengths[0]
        if past_len + input_ids.shape[1] > self.cfg.max_len:
            raise ValueError('历史长度 + 本次新增长度超过上下文；本课不截断/滑动/重置位置')
        return past_len

    def forward_cached(self, input_ids: torch.Tensor,
                       past_key_values: KVCache | None = None) -> tuple[torch.Tensor, KVCache]:
        """输入仅包含新增 token；返回本次 logits [B,Q,V] 与每层扩展后的 cache。"""
        past_len = self._validate_cached_inputs(input_ids, past_key_values)
        q_len = input_ids.shape[1]
        positions = torch.arange(past_len, past_len + q_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        updated = []
        for layer, block in enumerate(self.blocks):
            old_pair = None if past_key_values is None else past_key_values[layer]
            attention_output, new_pair = cached_attention(block.attn, block.norm1(x), old_pair)
            x = x + block.residual_dropout(attention_output)
            x = x + block.residual_dropout(block.ffn(block.norm2(x)))
            updated.append(new_pair)
        return self.lm_head(self.final_norm(x)), tuple(updated)

In [9]:
set_seed(19)
probe_cfg = GPTConfig(vocab_size=9, max_len=16, d_model=16,
                      n_heads=4, n_layers=2, ff_dim=32, dropout=0.1)
probe = CachedTinyGPT(probe_cfg).to(DEVICE).eval()
probe_ids = torch.tensor([[1, 4, 5, 6, 7, 8]], device=DEVICE)

with torch.no_grad():
    prefix_logits, prefix_cache = probe.forward_cached(probe_ids[:, :3])
    next_logits, extended_cache = probe.forward_cached(probe_ids[:, 3:4], prefix_cache)

print("prefill logits:", tuple(prefix_logits.shape))
print("第一层 prefill K:", tuple(prefix_cache[0][0].shape))
print("单步 logits:", tuple(next_logits.shape))
print("第一层 decode K:", tuple(extended_cache[0][0].shape))
print("旧 cache 仍长:", prefix_cache[0][0].shape[-2])
print("cache 中张量 requires_grad:", extended_cache[0][0].requires_grad)

prefill logits: (1, 3, 9)
第一层 prefill K: (1, 4, 3, 4)
单步 logits: (1, 1, 9)
第一层 decode K: (1, 4, 4, 4)
旧 cache 仍长: 3
cache 中张量 requires_grad: False


## 5. 先核对数学结果，再测速度

只看生成出的文字相同是不够的：分数算错了，最大值位置仍可能碰巧相同。
我们会核对**全部位置的原始 logits**，再核对最终每层 K/V：

- 一次处理完整序列；
- 每次只处理一个新 token；
- 分成 `[3,2,1]` 这样的多个 chunk。

不同矩阵形状可能改变浮点累加顺序，所以使用误差容限，而非要求逐位相等。
本课 FP32 检查设置 `rtol=2e-4, atol=2e-5`。这不是任何精度、任何模型都适用的万能阈值。
即使 logits 接近，极接近并列的最大值仍可能因微小误差导致后续生成分叉。

本节先用随机模型验证机制，验证缓存不需要先让模型会写中文。

In [10]:
def cache_payload_bytes(cache: KVCache | None) -> int:
    """K/V 有效元素载荷；不是进程内存、分配器预留量或 GPU 峰值显存。"""
    if cache is None:
        return 0
    return sum(t.numel() * t.element_size() for pair in cache for t in pair)


def expected_cache_bytes(cfg: GPTConfig, batch_size: int, tokens: int, bytes_per_element: int) -> int:
    for name, value in [('batch_size', batch_size), ('bytes_per_element', bytes_per_element)]:
        positive_int(value, name)
    if type(tokens) is not int or tokens < 0:
        raise ValueError('tokens 必须是非负整数')
    # MHA 中 H_kv * d_head = d_model。因子2来自 K 和 V，不是2个头。
    return 2 * cfg.n_layers * batch_size * tokens * cfg.d_model * bytes_per_element


def compare_chunked(model: CachedTinyGPT, ids: torch.Tensor, chunks: list[int]) -> dict:
    if (not isinstance(chunks, list) or not chunks or any(type(c) is not int or c <= 0 for c in chunks)
            or sum(chunks) != ids.shape[1]):
        raise ValueError('chunks 必须是正整数列表，长度总和等于输入 T')
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            full = model(ids)
            all_logits, all_cache = model.forward_cached(ids)
            cache, outputs, trace, start = None, [], [], 0
            for q_len in chunks:
                logits, cache = model.forward_cached(ids[:, start:start + q_len], cache)
                outputs.append(logits)
                trace.append({'past_tokens': start, 'new_tokens': q_len,
                              'cache_tokens': cache[0][0].shape[-2],
                              'first_layer_K_shape': list(cache[0][0].shape),
                              'cache_payload_bytes': cache_payload_bytes(cache)})
                start += q_len
            incremental = torch.cat(outputs, dim=1)
            torch.testing.assert_close(all_logits, full, rtol=2e-4, atol=2e-5)
            torch.testing.assert_close(incremental, full, rtol=2e-4, atol=2e-5)
            for actual, expected in zip(cache, all_cache):
                for a, e in zip(actual, expected):
                    torch.testing.assert_close(a, e, rtol=2e-4, atol=2e-5)
            return {'chunks': chunks, 'max_abs_logit_diff': float((incremental-full).abs().max().item()),
                    'full_prefill_max_abs_diff': float((all_logits-full).abs().max().item()),
                    'cache_payload_bytes': cache_payload_bytes(cache), 'trace': trace,
                    'tolerance': {'rtol': 2e-4, 'atol': 2e-5}}
    finally:
        model.train(was_training)

In [11]:
for chunks in [[6], [1, 1, 1, 1, 1, 1], [3, 2, 1]]:
    comparison = compare_chunked(probe, probe_ids, chunks)
    print("分块:", chunks, "max_abs_diff:", comparison["max_abs_logit_diff"])
    for row in comparison["trace"]:
        print("  ", row)

分块: [6] max_abs_diff: 0.0
   {'past_tokens': 0, 'new_tokens': 6, 'cache_tokens': 6, 'first_layer_K_shape': [1, 4, 6, 4], 'cache_payload_bytes': 1536}
分块: [1, 1, 1, 1, 1, 1] max_abs_diff: 2.2351741790771484e-08
   {'past_tokens': 0, 'new_tokens': 1, 'cache_tokens': 1, 'first_layer_K_shape': [1, 4, 1, 4], 'cache_payload_bytes': 256}
   {'past_tokens': 1, 'new_tokens': 1, 'cache_tokens': 2, 'first_layer_K_shape': [1, 4, 2, 4], 'cache_payload_bytes': 512}
   {'past_tokens': 2, 'new_tokens': 1, 'cache_tokens': 3, 'first_layer_K_shape': [1, 4, 3, 4], 'cache_payload_bytes': 768}
   {'past_tokens': 3, 'new_tokens': 1, 'cache_tokens': 4, 'first_layer_K_shape': [1, 4, 4, 4], 'cache_payload_bytes': 1024}
   {'past_tokens': 4, 'new_tokens': 1, 'cache_tokens': 5, 'first_layer_K_shape': [1, 4, 5, 4], 'cache_payload_bytes': 1280}
   {'past_tokens': 5, 'new_tokens': 1, 'cache_tokens': 6, 'first_layer_K_shape': [1, 4, 6, 4], 'cache_payload_bytes': 1536}
分块: [3, 2, 1] max_abs_diff: 4.470348358154297e-08

### 5.1 为什么只有单 token 测试还不够？

对单个 query，如果把 K 和 V 按**相同顺序一起重排**，且所有 key 都被允许读取，
加权求和可能不变。因此，“K/V 追加顺序反了”的 bug，可能躲过单 token 输出测试。

但多个新 token 的 chunk 需要在内部进行因果遮挡；错误排列会让 mask 对错位置。
本课同时检查 chunk、每层 cache 数值和未来修改干预，不能只拿一句生成文字当验证。

另一个实验：两个不同请求交错前向，各自持有独立 cache；若把缓存放进模型上的一个共享全局变量，
就可能出现请求串扰。本实现把 cache 作为显式参数传递。

In [12]:
with torch.no_grad():
    reference = probe(probe_ids)
    changed = probe_ids.clone()
    changed[:, -1] = 4
    changed_logits, _ = probe.forward_cached(changed[:, 3:], prefix_cache)
    print("修改未来后，前两个新增位置最大差异:",
          float((changed_logits[:, :2] - reference[:, 3:5]).abs().max()))

    # 同一前缀分成两个分支，旧cache不被改写。
    branch_a = probe_ids[:, 3:4]
    branch_b = branch_a.new_tensor([[8]])
    out_a, cache_a = probe.forward_cached(branch_a, prefix_cache)
    out_b, cache_b = probe.forward_cached(branch_b, prefix_cache)
    print("旧长度 / 两个分支长度:", prefix_cache[0][0].shape[-2],
          cache_a[0][0].shape[-2], cache_b[0][0].shape[-2])

修改未来后，前两个新增位置最大差异: 1.4901161193847656e-08
旧长度 / 两个分支长度: 3 4 4


## 6. 内存：省计算，增加与历史相关的状态

普通 MHA、各层长度相同、KV 同精度且未量化时：

$$\text{KV bytes}=2\times L\times B\times T\times H_{kv}\times d_{head}\times s$$

$L$ 是层数，$B$ 是批大小，$T$ 是已缓存长度，$s$ 是每元素字节数。
因子 2 来自 **K 和 V 两份状态**。本课 $H_{kv}=H$，因此 $H_{kv}d_{head}=D$。

例如：2 层、D=128、B=1、T=143、FP32 每元素 4 字节：

$$2\times2\times1\times143\times128\times4=292864\text{ bytes}=286\text{ KiB}$$

这是**缓存有效元素载荷**，不包括模型权重、临时分数/输出、Python 对象、分配器预留，
也没有计入 `torch.cat` 过程中旧/新分配同时存在的峰值。

因此，`nvidia-smi` 的数值不会恰好等于这个公式；权重量化也不自动意味着 KV 已量化。
静态缓存、分页缓存、KV 量化和 GQA 会改变分配或存储方式，后续再学习。

参考：[Hugging Face cache strategies](https://huggingface.co/docs/transformers/kv_cache)。

In [13]:
byte_cfg = GPTConfig(vocab_size=41, max_len=256, d_model=128,
                     n_heads=4, n_layers=2, ff_dim=512, dropout=0.0)
for tokens in [16, 64, 143]:
    payload = expected_cache_bytes(byte_cfg, 1, tokens, 4)
    print(f"T={tokens:3d}: {payload:7d} bytes = {payload/1024:.1f} KiB")
print("probe缓存实测:", cache_payload_bytes(extended_cache), "bytes")
print("probe缓存公式:", expected_cache_bytes(probe_cfg, 1, 4, 4), "bytes")

T= 16:   32768 bytes = 32.0 KiB
T= 64:  131072 bytes = 128.0 KiB
T=143:  292864 bytes = 286.0 KiB
probe缓存实测: 1024 bytes
probe缓存公式: 1024 bytes


## 7. 把 cache 接回生成循环

第一轮传整个提示；之后只传刚选出的一个 token。
每次都用本次 logits 的**最后一个位置**选择下一个 token。

下面的生成函数只做贪心选择，目的是隔离缓存对照，不是重复上一课所有采样策略。
BOS/PAD 不作为生成候选；EOS、预算和上下文长度决定何时停止。

本接口与前课一致，将返回文本的总 token 数限制在 `max_len`。
若提示已经长达 `max_len`，生成接口不再追加；底层 `forward_cached` 仍允许计算恰好 `max_len` 的输入。

返回值只报告 cache 长度与载荷，不把它伪装成可直接断点续写的完整会话状态。
最后选出的 token 通常尚未经过前向，继续生成时还需要保存并处理这枚待输入 token。

In [14]:
def generate_greedy(model: CachedTinyGPT, vocab: list[str], prompt: str = '', *,
                    use_cache: bool = True, max_new_tokens: int = 24) -> dict:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size or not isinstance(prompt, str):
        raise ValueError('词表和模型不一致，或 prompt 不是字符串')
    if type(use_cache) is not bool or type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError('use_cache 必须是布尔值；max_new_tokens 必须是非负整数')
    ids = [BOS_ID] + encode(prompt, {t: i for i, t in enumerate(vocab)})
    if len(ids) > model.cfg.max_len:
        raise ValueError('提示超过上下文；不能静默截断')
    device = next(model.parameters()).device
    current = torch.tensor([ids], device=device, dtype=torch.long)
    cache, trace, new_ids = None, [], []
    reason = 'max_new_tokens' if max_new_tokens == 0 else 'context_limit'
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for step in range(max_new_tokens):
                # 与前课一致：本接口将返回的总长度限制在 max_len，不多吐出第 max_len+1 个编号。
                if current.shape[1] >= model.cfg.max_len:
                    reason = 'context_limit'
                    break
                fed = current if not use_cache or cache is None else current[:, -1:]
                if use_cache:
                    logits, cache = model.forward_cached(fed, cache)
                else:
                    logits = model(fed)
                last = logits[:, -1, :].clone()
                last[:, [BOS_ID, PAD_ID]] = float('-inf')
                next_id = int(last.argmax(-1).item())
                new_ids.append(next_id)
                trace.append({'prediction_step': step, 'input_tokens_this_call': fed.shape[1],
                              'processed_prefix_tokens': current.shape[1],
                              'cache_tokens_after_call': 0 if cache is None else cache[0][0].shape[-2],
                              'selected_token_id': next_id})
                current = torch.cat((current, current.new_tensor([[next_id]])), dim=1)
                if next_id == EOS_ID:
                    reason = 'eos'
                    break
                reason = 'max_new_tokens'
    finally:
        model.train(was_training)
    visible = ''.join(vocab[i] for i in new_ids if i not in (BOS_ID, EOS_ID, PAD_ID))
    return {'prompt': prompt, 'text': prompt + visible, 'use_cache': use_cache,
            'new_ids': new_ids, 'stop_reason': reason, 'trace': trace,
            'cache_tokens_at_return': 0 if cache is None else cache[0][0].shape[-2],
            'cache_payload_bytes': cache_payload_bytes(cache),
            'note': '最后选出的 token 尚未送回模型，因此通常未进入 cache；cache 不作为可续接状态返回。'}

## 8. 测量方法：不要拿不同长度的生成互相比速度

若无缓存生成 20 个 token，有缓存恰好 5 个 token 就遇到 EOS，比较总耗时是不公平的。
本课使用**固定 token replay**：预先给定相同提示与后续 token，两条路径输出相同数量的预测。

专用性能模型：2 层、D=128、4 头、FFN=512、FP32、随机固定权重。
它与教学语言模型分开，避免为测试长前缀而擅自扩充已训练的绝对位置表。
无须训练这个模型，因为只测相同计算的两种实现，不测语言质量。

- 提示长度：16、64、128。
- 每次：16 个预测，即 1 次 prefill + 15 次 decode。
- 预热 2 轮，重复 7 轮，交替先测哪条路径，报告中位数。
- CUDA 在计时边界同步；否则 CPU 计时可能只测到了异步提交。
- 两条路径都验证结果；不把 cache 必须更快写成测试断言。

本课 `prefill_ms` 只包含模型到 logits，不是含分词、排队和网络的服务端 TTFT。
`decode_ms_per_prediction` 是一组 decode 总耗时除以次数，不是逐 token 的 p95 延迟。
参数检查、Python 和动态 `cat` 开销都包含在时间内，因此这是教学实现的微基准。

参考：[CUDA synchronize](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)、
[CUDA 异步执行说明](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution)。

In [15]:
def synchronize(device: torch.device) -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def _replay_path(model: CachedTinyGPT, ids: torch.Tensor, prefix_len: int, predictions: int,
                 cached: bool) -> tuple[dict, torch.Tensor]:
    """内部函数。固定 token replay，不受 EOS、文本差异或采样影响。
    N 个预测 = 1 次 prefill + N-1 次 decode；后续 token 使用预先生成的 ids。
    """
    device = ids.device
    synchronize(device)
    start = time.perf_counter()
    if cached:
        logits, cache = model.forward_cached(ids[:, :prefix_len])
    else:
        logits, cache = model(ids[:, :prefix_len]), None
    first = logits[:, -1, :].clone()
    synchronize(device)
    prefill_end = time.perf_counter()
    outputs = [first]
    for step in range(1, predictions):
        end = prefix_len + step
        if cached:
            logits, cache = model.forward_cached(ids[:, end-1:end], cache)
        else:
            logits = model(ids[:, :end])
        outputs.append(logits[:, -1, :].clone())
    synchronize(device)
    end_time = time.perf_counter()
    result = {'prefill_ms': 1000*(prefill_end-start),
              'decode_total_ms': 1000*(end_time-prefill_end),
              'total_ms': 1000*(end_time-start),
              'cache_payload_bytes': cache_payload_bytes(cache)}
    # stack/转CPU不在计时段中；两条路径保留相同数量的末位置 logits。
    return result, torch.stack(outputs, dim=1)


def benchmark_cache(device: torch.device, *, repeats: int = 7, warmups: int = 2,
                    prefixes: tuple[int, ...] = (16, 64, 128), predictions: int = 16) -> dict:
    positive_int(repeats, 'repeats')
    positive_int(warmups, 'warmups')
    if type(predictions) is not int or predictions < 2:
        raise ValueError('predictions 至少为2，以便测量decode')
    if not prefixes or any(type(p) is not int or p <= 0 for p in prefixes):
        raise ValueError('prefixes 必须包含正整数')
    # 专用随机模型，不改变加载/训练模型的位置表；不是语言质量实验。
    cfg = GPTConfig(vocab_size=41, max_len=max(prefixes)+predictions,
                    d_model=128, n_heads=4, n_layers=2, ff_dim=512, dropout=0.0)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(1900)
        model = CachedTinyGPT(cfg).to(device).eval()
    rows = []
    generator = torch.Generator().manual_seed(1901)
    with torch.no_grad():
        for prefix in prefixes:
            sequence = torch.randint(4, cfg.vocab_size, (1, prefix+predictions-1), generator=generator).to(device)
            for _ in range(warmups):
                _replay_path(model, sequence, prefix, predictions, False)
                _replay_path(model, sequence, prefix, predictions, True)
            measurements = {'full': [], 'cache': []}
            max_error = 0.0
            for repeat in range(repeats):
                results = {}
                # 交替先测哪条路径，减少固定顺序偏差。
                for cached in ([False, True] if repeat % 2 == 0 else [True, False]):
                    key = 'cache' if cached else 'full'
                    timings, logits = _replay_path(model, sequence, prefix, predictions, cached)
                    measurements[key].append(timings)
                    results[key] = logits
                torch.testing.assert_close(results['full'], results['cache'], rtol=2e-4, atol=2e-5)
                max_error = max(max_error, float((results['full']-results['cache']).abs().max().item()))
            row = {'prefix_tokens': prefix, 'predictions': predictions, 'decode_calls': predictions-1,
                   'max_abs_logit_diff': max_error, 'raw_measurements_ms': measurements}
            for key in ('full', 'cache'):
                row[key] = {name: statistics.median(m[name] for m in measurements[key])
                            for name in ('prefill_ms', 'decode_total_ms', 'total_ms')}
                row[key]['decode_ms_per_prediction'] = row[key]['decode_total_ms']/(predictions-1)
            row['decode_speedup'] = row['full']['decode_total_ms']/row['cache']['decode_total_ms']
            row['total_speedup'] = row['full']['total_ms']/row['cache']['total_ms']
            row['full_processed_token_positions'] = predictions*prefix+predictions*(predictions-1)//2
            row['cached_processed_token_positions'] = prefix+predictions-1
            row['cache_payload_bytes'] = measurements['cache'][0]['cache_payload_bytes']
            row['expected_cache_bytes'] = expected_cache_bytes(cfg, 1, prefix+predictions-1, 4)
            if row['cache_payload_bytes'] != row['expected_cache_bytes']:
                raise AssertionError('缓存载荷与形状公式不一致')
            rows.append(row)
    return {'config': asdict(cfg), 'parameters': count_parameters(model),
            'model': 'fixed_random_model_for_runtime_only', 'dtype': 'float32',
            'batch_size': 1, 'repeats': repeats, 'warmups': warmups, 'rows': rows,
            'notes': ['同一设备、同一组权重、相同已知token路径；不做文字质量评测',
                      'CPU单线程；CUDA计时边界同步；包括Python、参数检查与动态cat开销',
                      'prefill只计模型到logits，不是含排队/网络/分词的服务端TTFT',
                      'decode每预测耗时=一段decode耗时/(N-1)，不是逐token延迟分位数',
                      '缓存字节是有效K/V元素载荷，不是进程内存或GPU峰值显存',
                      '不同机器/上下文/内核结果可能不同，不要求cache一定更快']}

### 8.1 计算量与墙钟时间不是同一个量

提示 P、输出 N，在本课“完整前缀重算”基线下，所有层各自处理的位置总数：

$$\text{full}=NP+\frac{N(N-1)}2,\qquad \text{cache}=P+N-1$$

这里计算的是投影/FFN 等逐位置工作的输入位置数，不是 FLOPs，也不是速度比。
每头注意力分数元素数大致从：

$$\sum_{i=0}^{N-1}(P+i)^2\quad\rightarrow\quad P^2+\sum_{i=1}^{N-1}(P+i)$$

decode 仍要读取越来越长的历史，**并没有变成与上下文长度无关的常数开销**。
GPU 利用率、内存读取、内核启动、校验与分配也会影响真实耗时。
只要实测方法公平，短输入时看到提升很小甚至变慢，也应如实记录。

## 9. 完整实验：准备一份固定模型，核对生成，再跑基准

下面两格复用前课的数据、损失、训练和检查点工具，作为独立运行的支持代码。
新逻辑已经在前面讲完，复用部分不要求重新背写。

默认训练 40 轮，按验证损失选权重；这只是模板语料，不是能力评测。
`CHECKPOINT` 指向第 17/18/19 课检查点时跳过训练。
所有缓存只在推理函数里创建，训练仍使用全量前向。

In [16]:
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]


def ensure_demo_file(path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("x", encoding="utf-8") as handle:
            handle.write("\n".join(demo_lines()) + "\n")
    except FileExistsError:
        pass
    return path


def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts


def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]


def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token


def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]


class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]


def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))


def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count


def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}


def evaluate(model: TinyGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)


def train_epoch(model: TinyGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。

In [17]:
def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def save_checkpoint(path: Path, model: TinyGPT, id_to_token: list[str], best_epoch: int) -> None:
    validate_vocab(id_to_token)
    if len(id_to_token) != model.cfg.vocab_size:
        raise ValueError("模型与词表大小不一致")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format_version": 1, "config": asdict(model.cfg), "vocab": id_to_token,
                "model_state": snapshot_state(model), "best_epoch": best_epoch,
                "purpose": "inference_only_not_exact_training_resume"}, path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[TinyGPT, list[str], dict]:
    # 只加载自己生成/信任的文件；weights_only 也不是任意文件安全性的保证。
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("format_version") != 1:
        raise ValueError("不支持的检查点格式")
    vocab = payload["vocab"]
    validate_vocab(vocab)
    cfg = GPTConfig(**payload["config"])
    if len(vocab) != cfg.vocab_size:
        raise ValueError("检查点词表与配置不一致")
    # 构造随机初始化模型不消耗调用者的 CPU 随机状态。
    with torch.random.fork_rng(devices=[]):
        model = CachedTinyGPT(cfg)
    model.load_state_dict(payload["model_state"], strict=True)
    model.to(device).eval()
    return model, vocab, {"best_epoch": payload["best_epoch"], "purpose": payload["purpose"]}


def train_demo_model(device: torch.device, *, epochs: int = 40, seed: int = 17) -> tuple[TinyGPT, list, dict]:
    positive_int(epochs, "epochs")
    train_texts, val_texts = split_texts(demo_lines(), seed)
    mapping, vocab = build_vocab(train_texts)
    cfg = GPTConfig(len(vocab))
    train_data = CharDataset(train_texts, mapping, cfg.max_len)
    val_data = CharDataset(val_texts, mapping, cfg.max_len)
    train_loader = make_loader(train_data, 16, True, seed)
    train_eval = make_loader(train_data, 16)
    val_loader = make_loader(val_data, 16)
    set_seed(seed)
    model = CachedTinyGPT(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    # 与第17课保持一致的初始化、数据遍历及验证选择。
    before = {"train": evaluate(model, train_eval, device), "validation": evaluate(model, val_loader, device)}
    best_loss, best_epoch, best_state = before["validation"]["loss"], 0, snapshot_state(model)
    history = []
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        train_metrics = evaluate(model, train_eval, device)
        validation = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train": train_metrics, "validation": validation,
                        "online_train": online})
        if validation["loss"] < best_loss:
            best_loss, best_epoch, best_state = validation["loss"], epoch, snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"准备固定模型 epoch={epoch:02d} train={train_metrics['loss']:.4f} val={validation['loss']:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    # 清掉训练残留梯度；比较过程不再创建/更新它们。
    model.zero_grad(set_to_none=True)
    return model, vocab, {"source": "embedded_demo_training", "epochs": epochs, "seed": seed,
                          "train_texts": train_texts, "validation_texts": val_texts,
                          "before": before, "best_epoch": best_epoch, "history": history}


def model_fingerprint(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str((tuple(value.shape), value.dtype)).encode('utf-8'))
        digest.update(bytes(value.reshape(-1).view(torch.uint8).tolist()))
    return digest.hexdigest()


def run_experiment(output_dir: Path, *, device: torch.device = torch.device('cpu'),
                   checkpoint: Path | None = None, epochs: int = 40, repeats: int = 7,
                   skip_benchmark: bool = False) -> dict:
    output_dir = Path(output_dir)
    names = ['frozen_model.pt', 'training.json', 'correctness.json', 'generations.json', 'benchmark.json', 'experiment.json']
    if checkpoint is not None and any(Path(checkpoint).resolve() == (output_dir/n).resolve() for n in names):
        raise ValueError('输出不能覆盖输入检查点；请换 output-dir')
    if checkpoint is None:
        model, vocab, training_info = train_demo_model(device, epochs=epochs)
    else:
        model, vocab, training_info = load_checkpoint(Path(checkpoint), device)
        training_info.update({'source': str(Path(checkpoint).resolve()), 'training_skipped': True})
    model.eval()
    before = model_fingerprint(model)
    # 仅做计算等价性；即使自带词表不同，普通编号序列仍可验证机制。
    length = min(12, model.cfg.max_len)
    ids = torch.tensor([[BOS_ID] + [4 + i % (len(vocab)-4) for i in range(length-1)]], device=device)
    splits = [[length], [1]*length]
    if length >= 5:
        splits.append([3, 2, length-5] if length > 5 else [3, 2])
    correctness = {'comparisons': [compare_chunked(model, ids, c) for c in splits],
                   'sdpa_alignment': sdpa_alignment_demo(device)}
    candidates = ['', '我今天正在', '小明明天正在']
    prompts = [p for p in candidates if len(p)+1 <= model.cfg.max_len]
    generations = []
    for prompt in prompts:
        full = generate_greedy(model, vocab, prompt, use_cache=False)
        cached = generate_greedy(model, vocab, prompt, use_cache=True)
        same = full['new_ids'] == cached['new_ids'] and full['stop_reason'] == cached['stop_reason']
        # 记录实测，而非承诺任意近乎并列的分数下贪心必然逐位一致。
        generations.append({'prompt': prompt, 'same_ids_and_stop': same, 'full': full, 'cache': cached})
        print(f"生成一致={same} | {cached['text']!r} | cache长度={cached['cache_tokens_at_return']}")
    timings = None if skip_benchmark else benchmark_cache(device, repeats=repeats)
    after = model_fingerprint(model)
    if before != after:
        raise AssertionError('推理阶段改变了模型参数')
    output_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(output_dir/'frozen_model.pt', model, vocab, training_info.get('best_epoch', 0))
    reloaded, saved_vocab, _ = load_checkpoint(output_dir/'frozen_model.pt', device)
    if model_fingerprint(reloaded) != before or saved_vocab != vocab:
        raise AssertionError('检查点重载不一致')
    report = {'lesson': 19, 'torch_version': str(torch.__version__), 'python': platform.python_version(),
              'device': str(device), 'cpu_threads': torch.get_num_threads(),
              'cuda_device_name': torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
              'model_config': asdict(model.cfg), 'parameters': count_parameters(model),
              'weights_unchanged': True, 'fingerprint': before, 'reload_equal': True,
              'correctness': correctness, 'all_demo_greedy_equal': all(g['same_ids_and_stop'] for g in generations),
              'benchmark': timings,
              'limitations': ['等长无PAD的MHA动态缓存；无静态/分页/量化/GQA/RoPE/多请求调度',
                              '缓存必须对应同一权重和前缀；接口不识别语义上过期或串请求的缓存',
                              'cache不是长期记忆，不扩展位置表，也不是训练参数',
                              '浮点误差可能导致近并列处生成分叉；正确性先核对原始logits',
                              '性能实验使用独立随机模型；训练文本输出仅作机制演示']}
    for name, data in [('training.json', training_info), ('correctness.json', correctness),
                       ('generations.json', generations), ('benchmark.json', timings), ('experiment.json', report)]:
        (output_dir/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    if timings:
        for row in timings['rows']:
            print(f"P={row['prefix_tokens']:3d} | full decode={row['full']['decode_ms_per_prediction']:.3f}ms "
                  f"| cache={row['cache']['decode_ms_per_prediction']:.3f}ms "
                  f"| 本机比值={row['decode_speedup']:.2f}x | KV={row['cache_payload_bytes']/1024:.1f}KiB")
    print('权重未改变：True；输出目录：', output_dir.resolve())
    return report

In [18]:
report = run_experiment(
    OUTPUT_DIR,
    device=DEVICE,
    checkpoint=CHECKPOINT,
    epochs=EPOCHS,
    repeats=BENCHMARK_REPEATS,
    skip_benchmark=not RUN_BENCHMARK,
)
model, vocab, checkpoint_info = load_checkpoint(OUTPUT_DIR / "frozen_model.pt", DEVICE)
print("参数量:", report["parameters"])
print("权重未改变:", report["weights_unchanged"])
print("演示贪心一致:", report["all_demo_greedy_equal"])

准备固定模型 epoch=01 train=2.5748 val=2.6290


准备固定模型 epoch=10 train=0.4321 val=0.4717


准备固定模型 epoch=20 train=0.3962 val=0.4369


准备固定模型 epoch=30 train=0.3911 val=0.4255


准备固定模型 epoch=40 train=0.3896 val=0.4504
生成一致=True | '小明明天正在学习数学。' | cache长度=12
生成一致=True | '我今天正在学习数学。' | cache长度=11
生成一致=True | '小明明天正在学习数学。' | cache长度=12


P= 16 | full decode=1.254ms | cache=0.834ms | 本机比值=1.50x | KV=62.0KiB
P= 64 | full decode=1.807ms | cache=0.710ms | 本机比值=2.54x | KV=158.0KiB
P=128 | full decode=2.856ms | cache=0.692ms | 本机比值=4.13x | KV=286.0KiB
权重未改变：True；输出目录： /mnt/data/_lesson19_work/notebook_run/lesson_19_outputs
参数量: 109440
权重未改变: True
演示贪心一致: True


### 9.1 观察误差，不只看 True/False

查看不同分块的最大绝对误差，以及 cache 长度怎样增长。
`full_prefill_max_abs_diff` 是整段缓存前向与旧的全量前向之间的差异。
下面的误差来自本次模型和当前环境，不是保证所有平台都会得到完全相同的小数。

In [19]:
for item in report["correctness"]["comparisons"]:
    print("chunks =", item["chunks"])
    print("最大logit差异 =", item["max_abs_logit_diff"])
    print("最终KV载荷 =", item["cache_payload_bytes"], "bytes")

rows = json.loads((OUTPUT_DIR / "generations.json").read_text(encoding="utf-8"))
for row in rows:
    cached = row["cache"]
    print(); print("提示:", repr(row["prompt"]))
    print("无缓存:", row["full"]["text"])
    print("有缓存:", cached["text"])
    print("每步送入长度:", [step["input_tokens_this_call"] for step in cached["trace"]])
    print("已产生新token:", len(cached["new_ids"]),
          "最后cache长度:", cached["cache_tokens_at_return"],
          "停止:", cached["stop_reason"])

chunks = [12]
最大logit差异 = 0.0
最终KV载荷 = 12288 bytes
chunks = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
最大logit差异 = 1.9073486328125e-06
最终KV载荷 = 12288 bytes
chunks = [3, 2, 7]
最大logit差异 = 1.9073486328125e-06
最终KV载荷 = 12288 bytes

提示: ''
无缓存: 小明明天正在学习数学。
有缓存: 小明明天正在学习数学。
每步送入长度: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
已产生新token: 12 最后cache长度: 12 停止: eos

提示: '我今天正在'
无缓存: 我今天正在学习数学。
有缓存: 我今天正在学习数学。
每步送入长度: [6, 1, 1, 1, 1, 1]
已产生新token: 6 最后cache长度: 11 停止: eos

提示: '小明明天正在'
无缓存: 小明明天正在学习数学。
有缓存: 小明明天正在学习数学。
每步送入长度: [7, 1, 1, 1, 1, 1]
已产生新token: 6 最后cache长度: 12 停止: eos


### 9.2 读取本机性能结果

这里只列本次实验数据，不写固定“必然加速几倍”的承诺。
基准模型是随机权重；同一个基准模型的两条路径公平比较，不与训练模型的文字质量混合。
`raw_measurements_ms` 保存全部重复值，便于检查波动。

In [20]:
if report["benchmark"] is None:
    print("本次跳过性能基准。")
else:
    bench = report["benchmark"]
    print("基准模型配置:", bench["config"])
    print("参数量:", bench["parameters"], "重复:", bench["repeats"])
    print("P | full decode ms | cache decode ms | ratio | KV KiB | full/cache处理位置数")
    for row in bench["rows"]:
        print(f"{row['prefix_tokens']:3d} | "
              f"{row['full']['decode_ms_per_prediction']:.4f} | "
              f"{row['cache']['decode_ms_per_prediction']:.4f} | "
              f"{row['decode_speedup']:.2f}x | "
              f"{row['cache_payload_bytes']/1024:.1f} | "
              f"{row['full_processed_token_positions']}/{row['cached_processed_token_positions']}")

基准模型配置: {'vocab_size': 41, 'max_len': 144, 'd_model': 128, 'n_heads': 4, 'n_layers': 2, 'ff_dim': 512, 'dropout': 0.0}
参数量: 425728 重复: 7
P | full decode ms | cache decode ms | ratio | KV KiB | full/cache处理位置数
 16 | 1.2537 | 0.8336 | 1.50x | 62.0 | 376/31
 64 | 1.8066 | 0.7100 | 2.54x | 158.0 | 1144/79
128 | 2.8560 | 0.6923 | 4.13x | 286.0 | 2168/143


### 9.3 输出文件与边界

程序在当前工作目录下的 `lesson_19_outputs` 写入：

| 文件 | 内容 |
|---|---|
| `frozen_model.pt` | 固定权重、配置、词表；不含活跃请求 cache |
| `training.json` | 训练记录，或所加载检查点的信息 |
| `correctness.json` | 分块等价性、SDPA 对齐反例 |
| `generations.json` | 两条路径的完整文字、编号、停止原因和调用长度 |
| `benchmark.json` | 基准配置、每次耗时及汇总；跳过时为 null |
| `experiment.json` | 环境、权重指纹、完整报告和限制 |

同名输出覆盖，输入检查点不会覆盖。运行完整脚本还会保存 `checks.json`。
默认不生成数据包或整套学习包。

**不要跨请求误用 cache。** 本代码检查结构，但不会识别“这是另一个用户、另一份前缀或旧参数的缓存”。
修改权重、前文、设备、dtype 或位置策略后要重建；同一会话加入新 token 时才沿用同一历史。

这只是计算结果缓存，不是永久记忆。也没有实现静态预分配、分页、量化、长短不一请求合批、
滑动窗口或生产服务调度。

## 10. 自动检查

检查涵盖：偏移 mask、单步/多步/整段等价、每层 K/V 数值、因果性、分支与请求隔离、
缓存不修改权重、不污染 `state_dict`、无效参数、边界、缓存返回长度、生成模式恢复和保存重载。

检查不会要求“cache 一定更快”。性能是观测值，不是逻辑正确性的断言。
下面测试使用小随机模型，不依赖刚才训练的权重。

In [21]:
class Checks:
    def __init__(self) -> None:
        self.names: list[str] = []

    def ok(self, condition: bool, name: str) -> None:
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self, actual: torch.Tensor, expected: torch.Tensor, name: str,
              rtol: float = 2e-4, atol: float = 2e-5) -> None:
        torch.testing.assert_close(actual, expected, rtol=rtol, atol=atol, msg=name)
        self.names.append(name)

    def raises(self, function, name: str, error_type=ValueError) -> None:
        try:
            function()
        except error_type:
            self.names.append(name)
            return
        raise AssertionError(name + '：应报错却未报错')


def run_checks(device: torch.device = torch.device('cpu')) -> list[str]:
    c = Checks()
    mask = offset_causal_mask(3, 2, device)
    expected = torch.tensor([[1,1,1,1,0], [1,1,1,1,1]], dtype=torch.bool, device=device)[None,None]
    c.close(mask, expected, 'P=3,Q=2矩形掩码', rtol=0, atol=0)
    c.ok(bool(offset_causal_mask(5,1,device).all()), '单token能读取全部历史和自己')
    c.close(offset_causal_mask(0,3,device)[0,0], torch.ones(3,3,device=device,dtype=torch.bool).tril(), 'prefill退化为方形三角')
    for p,q in [(-1,1),(True,1),(1,0),(1,False)]:
        c.raises(lambda p=p,q=q: offset_causal_mask(p,q,device), f'拒绝mask参数{p!r},{q!r}')
    align = sdpa_alignment_demo(device)
    c.ok(math.isclose(align['upper_left_is_causal'],1.0,abs_tol=1e-6), '观察SDPA左上角对齐')
    c.ok(math.isclose(align['offset_mask'],2.5,abs_tol=1e-6), '偏移mask的正确手算结果')
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(19)
        cfg = GPTConfig(vocab_size=9,max_len=16,d_model=16,n_heads=4,n_layers=2,ff_dim=32,dropout=0.1)
        model = CachedTinyGPT(cfg).to(device).eval()
    ids = torch.tensor([[1,4,5,6,7,8],[1,8,7,6,5,4]],device=device)
    original_hash = model_fingerprint(model)
    original_keys = set(model.state_dict())
    c.ok(count_parameters(model)==parameter_formula(cfg), '缓存不增加参数量')
    c.raises(lambda: model.forward_cached(ids), '拒绝开启梯度的缓存调用')
    with torch.no_grad():
        model.train()
        c.raises(lambda: model.forward_cached(ids), '拒绝训练模式缓存')
        model.eval()
        full = model(ids)
        prefill, cache = model.forward_cached(ids[:,:3])
        c.close(prefill,full[:,:3], 'prefill与完整前缀输出一致')
        c.ok(len(cache)==cfg.n_layers,'每层独立K/V')
        c.ok(tuple(cache[0][0].shape)==(2,4,3,4),'缓存形状B,H,P,d')
        c.ok(all(not t.requires_grad and t.grad_fn is None for kv in cache for t in kv),'缓存不带求导图')
        c.ok(cache[0][0].data_ptr()!=cache[1][0].data_ptr(),'不同层不共用缓存张量')
        old = tuple(tuple(t.clone() for t in kv) for kv in cache)
        chunk_logits, extended = model.forward_cached(ids[:,3:],cache)
        c.close(chunk_logits, full[:,3:], '多token追加输出一致')
        c.ok(extended[0][0].shape[-2]==6,'缓存按序列轴增长')
        c.ok(cache[0][0].shape[-2]==3,'旧缓存长度不变')
        for layer,(kv,old_kv) in enumerate(zip(cache,old)):
            for side,(t,o) in enumerate(zip(kv,old_kv)):
                c.close(t,o,f'旧缓存未被改写 L{layer} side{side}',rtol=0,atol=0)
        one_shot, allcache=model.forward_cached(ids)
        c.close(one_shot,full,'一次缓存prefill与原forward一致')
        for layer,(a,b) in enumerate(zip(extended,allcache)):
            for side,(ta,tb) in enumerate(zip(a,b)):
                c.close(ta,tb,f'分块后的缓存数值等于全量 L{layer} side{side}')
        cur=None; sequence=[]
        for position in range(ids.shape[1]):
            logits,cur=model.forward_cached(ids[:,position:position+1],cur)
            sequence.append(logits)
        c.close(torch.cat(sequence,1),full,'逐token的每个logit一致')
        c.close(torch.cat(sequence,1).argmax(-1),full.argmax(-1),'本固定样例argmax一致',rtol=0,atol=0)
        c.ok(cache_payload_bytes(cur)==expected_cache_bytes(cfg,2,6,4),'测量缓存载荷等于公式')
        c.ok(cache_payload_bytes(None)==0,'空缓存载荷为0')
        c.ok(expected_cache_bytes(cfg,2,6,4)==2*expected_cache_bytes(cfg,1,6,4),'缓存随batch线性增长')
        c.ok(expected_cache_bytes(cfg,1,6,4)==2*expected_cache_bytes(cfg,1,3,4),'缓存随长度线性增长')
        c.ok(set(model.state_dict())==original_keys,'缓存不进入state_dict')
        changed=ids.clone(); changed[:,-1]=4
        changed_logits,_=model.forward_cached(changed[:,3:],cache)
        c.close(changed_logits[:,:-1],chunk_logits[:,:-1],'chunk内部未来不能影响过去')
        # 两个请求交错运行，各自持有自己的缓存。
        _,a=model.forward_cached(ids[:1,:2]); _,b=model.forward_cached(ids[1:,:2])
        la,_=model.forward_cached(ids[:1,2:4],a); lb,_=model.forward_cached(ids[1:,2:4],b)
        c.close(la,full[:1,2:4],'请求A缓存独立')
        c.close(lb,full[1:,2:4],'请求B缓存独立')
        # 同一个历史用于分叉，函数不原地破坏旧cache。
        branch=ids[:,:4].clone(); branch[:,-1]=8
        branch_logits,_=model.forward_cached(branch[:,3:],cache)
        c.close(branch_logits,model(branch)[:,-1:],'复用不可变历史分支计算正确')
        for name,bad in [('空输入',ids[:,:0]),('一维输入',ids[0]),('float编号',ids.float()),
                         ('负编号',ids.new_tensor([[-1]])),('越界编号',ids.new_tensor([[9]])),
                         ('PAD输入',ids.new_tensor([[1,3]]))]:
            c.raises(lambda bad=bad: model.forward_cached(bad),name)
        c.raises(lambda: model.forward_cached(ids[:,:1],()), '拒绝空tuple缓存')
        c.raises(lambda: model.forward_cached(ids[:,:1],cache[:1]), '拒绝层数不符')
        c.raises(lambda: model.forward_cached(ids[:1,:1],cache),'拒绝batch不符')
        wrong_dtype=tuple(tuple(t.double() for t in kv) for kv in cache)
        c.raises(lambda:model.forward_cached(ids[:,:1],wrong_dtype),'拒绝cache精度不符')
        wrong_len=(cache[0],tuple(t[:,:,:2] for t in cache[1]))
        c.raises(lambda:model.forward_cached(ids[:,:1],wrong_len),'拒绝各层长度不符')
        bad_grad=tuple(tuple(t.clone().requires_grad_() for t in kv) for kv in cache)
        c.raises(lambda:model.forward_cached(ids[:,:1],bad_grad),'拒绝保留梯度的cache')
        c.raises(lambda:model.forward_cached(ids[:,:1],((cache[0][0],),cache[1])),'拒绝缺少V')
        bad_v=((cache[0][0],cache[0][1][:,:,:2]),cache[1])
        c.raises(lambda:model.forward_cached(ids[:,:1],bad_v),'拒绝K/V形状不同')
        c.raises(lambda:model.forward_cached(ids.new_full((2,14),4),cache),'拒绝past+new超长')
        _,boundary=model.forward_cached(ids.new_full((2,16),4))
        c.ok(boundary[0][0].shape[-2]==16,'恰好max_len可前向')
        c.raises(lambda:model.forward_cached(ids[:,:1],boundary),'边界后不能继续追加')
        with torch.autocast(device_type=device.type, enabled=True):
            c.raises(lambda:model.forward_cached(ids[:,:1]),'明确拒绝autocast')
    c.ok(model_fingerprint(model)==original_hash,'缓存推理不更新参数')
    # 全量训练仍可反向传播，不需要拆开序列保留cache。
    model.train(); model.zero_grad(set_to_none=True)
    model(ids).square().mean().backward()
    c.ok(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()),'普通forward仍可训练')
    model.zero_grad(set_to_none=True); model.eval()
    vocab=SPECIAL_TOKENS+list('我爱学习。')
    full_text=generate_greedy(model,vocab,'我',use_cache=False,max_new_tokens=5)
    cache_text=generate_greedy(model,vocab,'我',use_cache=True,max_new_tokens=5)
    c.ok(full_text['new_ids']==cache_text['new_ids'],'固定模型贪心生成一致')
    c.ok(full_text['stop_reason']==cache_text['stop_reason'],'生成停止原因一致')
    if cache_text['new_ids']:
        c.ok(cache_text['cache_tokens_at_return']==2+len(cache_text['new_ids'])-1,'最后生成token尚未进入缓存')
        c.ok([t['input_tokens_this_call'] for t in cache_text['trace']]==[2]+[1]*(len(cache_text['new_ids'])-1),'prefill一次后只传新增token')
    zero=generate_greedy(model,vocab,'我',max_new_tokens=0)
    c.ok(not zero['trace'] and zero['cache_tokens_at_return']==0,'零预算不前向不建cache')
    no_room=generate_greedy(model,vocab,'我'*15,max_new_tokens=2)
    c.ok(no_room['stop_reason']=='context_limit' and not no_room['new_ids'],'生成接口的上下文边界')
    c.raises(lambda:generate_greedy(model,vocab,'我'*16),'拒绝超长提示')
    c.raises(lambda:generate_greedy(model,vocab,'我',max_new_tokens=-1),'拒绝负生成预算')
    model.train(); generate_greedy(model,vocab,'我',max_new_tokens=1)
    c.ok(model.training,'生成后恢复模型模式')
    model.eval()
    # 随机模型也可以做固定时长回放；不把EOS导致的提前结束误算成速度收益。
    with torch.no_grad():
        a,la=_replay_path(model,ids[:1,:5],2,4,False)
        b,lb=_replay_path(model,ids[:1,:5],2,4,True)
        c.close(la,lb,'基准固定路径所有预测一致')
        c.ok(a['total_ms']>0 and b['total_ms']>0,'计时为正，不假定谁更快')
    c.raises(lambda:compare_chunked(model,ids,[2,2]),'拒绝chunk长度不匹配')
    c.raises(lambda:expected_cache_bytes(cfg,1,-1,4),'拒绝负缓存长度')
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp)/'model.pt'
        save_checkpoint(path,model,vocab,0)
        restored,restored_vocab,_=load_checkpoint(path,device)
        c.ok(restored_vocab==vocab and model_fingerprint(restored)==model_fingerprint(model),'旧格式检查点保存重载兼容')
        with torch.no_grad():
            r,_=restored.forward_cached(ids)
            c.close(r,model(ids),'重载模型的缓存路径一致')
    print(f'通过固定检查：{len(c.names)} 项')
    return c.names

In [22]:
check_names = run_checks(DEVICE)
print("最后5项:", check_names[-5:])

通过固定检查：75 项
最后5项: ['计时为正，不假定谁更快', '拒绝chunk长度不匹配', '拒绝负缓存长度', '旧格式检查点保存重载兼容', '重载模型的缓存路径一致']


## 11. 独立练习：先停在这里，再看答案

### 练习 A：手写偏移因果 mask

实现 `offset_mask_ex(past_len, new_len, device)`，返回 `[1,1,Q,P+Q]` 的 bool 张量。
要检查 `past_len>=0`、`new_len>0`，且拒绝布尔值冒充整数。

先手写 P=2、Q=3 的矩阵，再写代码。不要调用已有的 `offset_causal_mask()` 代替实现。

In [23]:
def check_exercise_a(function):
    expected = torch.tensor([
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1],
    ], dtype=torch.bool)[None, None]
    torch.testing.assert_close(function(2, 3, "cpu"), expected)
    assert bool(function(7, 1, "cpu").all())
    torch.testing.assert_close(function(0, 3, "cpu")[0, 0],
                               torch.ones(3, 3, dtype=torch.bool).tril())
    for p, q in [(-1, 1), (0, 0), (True, 1)]:
        try:
            function(p, q, "cpu")
        except ValueError:
            continue
        raise AssertionError("没有拒绝错误参数")
    print("练习 A 通过")


def offset_mask_ex(past_len, new_len, device="cpu"):
    # TODO: 构造 query 的绝对位置，与所有 key 的位置比较。
    raise NotImplementedError("请补全练习 A")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(offset_mask_ex)

### 练习 B：正确追加 K/V，并保留旧历史

实现 `append_kv_ex(past, k_new, v_new)`：
历史为空时返回新 K/V；否则沿序列轴追加。
要求 K/V 是相同形状的四维张量，新长度非零，旧/新 B、H、d、dtype、device 对得上。
不得原地改写旧 cache；不能把新 token 放在历史之前。

这道题不做注意力计算，先把存储行为写正确。

In [24]:
def check_exercise_b(function):
    old_k = torch.arange(12, dtype=torch.float32).reshape(1, 2, 3, 2)
    old_v = old_k + 100
    new_k = torch.full((1, 2, 2, 2), 50.0)
    new_v = new_k + 100
    before_k, before_v = old_k.clone(), old_v.clone()
    k, v = function((old_k, old_v), new_k, new_v)
    assert k.shape == (1, 2, 5, 2) and v.shape == k.shape
    torch.testing.assert_close(k[:, :, :3], before_k)
    torch.testing.assert_close(v[:, :, :3], before_v)
    torch.testing.assert_close(k[:, :, 3:], new_k)
    torch.testing.assert_close(v[:, :, 3:], new_v)
    torch.testing.assert_close(old_k, before_k)
    torch.testing.assert_close(old_v, before_v)
    a, b = function(None, new_k, new_v)
    torch.testing.assert_close(a, new_k)
    torch.testing.assert_close(b, new_v)
    try:
        function((old_k, old_v), new_k.expand(2, -1, -1, -1), new_v.expand(2, -1, -1, -1))
    except ValueError:
        pass
    else:
        raise AssertionError("batch不一致应被拒绝")
    print("练习 B 通过")


def append_kv_ex(past, k_new, v_new):
    # TODO: 校验后沿 dim=-2 追加，返回 (K,V)。
    raise NotImplementedError("请补全练习 B")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(append_kv_ex)

### 练习 C：使用 cache 按块读取整段，并还原全部 logits

实现 `incremental_logits_ex(model, ids, chunks)`：
按 chunk 长度切出尚未处理的部分，依次调用 `forward_cached`，把每块的 logits 沿时间轴拼接。
只处理输入，不做抽样。检查 chunk 长度总和正确，使用 eval/no_grad，并恢复调用前的模型模式。

不能直接调用 `model(ids)` 返回答案，也不能每次重新传全部前缀。
检查器会查看每次缓存前向收到的长度，同时比较每个位置的 logits。

In [25]:
def check_exercise_c(function):
    from unittest import mock
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(1919)
        cfg = GPTConfig(9, max_len=16, d_model=16, n_heads=4,
                        n_layers=2, ff_dim=32, dropout=0.2)
        m = CachedTinyGPT(cfg).eval()
    ids = torch.tensor([[1, 4, 5, 6, 7, 8]], dtype=torch.long)
    with torch.no_grad():
        expected = m(ids)
    m.train()
    with mock.patch.object(m, "forward_cached", wraps=m.forward_cached) as spy:
        actual = function(m, ids, [2, 3, 1])
        lengths = [call.args[0].shape[1] for call in spy.call_args_list]
    assert lengths == [2, 3, 1], lengths
    assert m.training, "调用后应恢复训练模式"
    assert not actual.requires_grad
    torch.testing.assert_close(actual, expected, rtol=2e-4, atol=2e-5)
    try:
        function(m, ids, [1, 1])
    except ValueError:
        pass
    else:
        raise AssertionError("chunk总长度错误应拒绝")
    print("练习 C 通过")


def incremental_logits_ex(model, ids, chunks):
    # TODO: 增量前向，拼接所有位置的logits；finally恢复模式。
    raise NotImplementedError("请补全练习 C")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(incremental_logits_ex)

## 12. 参考答案

先尝试独立完成上面三题。下面代码能运行，只说明参考实现通过检查，**不代表你的练习已经完成**。

### A：绝对位置比较

In [26]:
def offset_mask_answer(past_len, new_len, device="cpu"):
    if type(past_len) is not int or past_len < 0:
        raise ValueError("past_len需要非负整数")
    if type(new_len) is not int or new_len <= 0:
        raise ValueError("new_len需要正整数")
    query_pos = torch.arange(past_len, past_len + new_len, device=device)
    key_pos = torch.arange(past_len + new_len, device=device)
    return (key_pos[None, :] <= query_pos[:, None])[None, None]

check_exercise_a(offset_mask_answer)

练习 A 通过


### B：沿序列轴追加

这里只实现练习要求的局部校验；完整模型还要检查各层长度和模型上下文。

In [27]:
def append_kv_answer(past, k_new, v_new):
    for tensor in (k_new, v_new):
        if not isinstance(tensor, torch.Tensor) or tensor.ndim != 4 or min(tensor.shape) <= 0:
            raise ValueError("新K/V必须是非空四维张量")
    if k_new.shape != v_new.shape or k_new.device != v_new.device or k_new.dtype != v_new.dtype:
        raise ValueError("新K/V不匹配")
    if past is None:
        return k_new, v_new
    if not isinstance(past, tuple) or len(past) != 2:
        raise ValueError("past必须是(K,V)")
    for old, new in zip(past, (k_new, v_new)):
        if not isinstance(old, torch.Tensor) or old.ndim != 4 or min(old.shape) <= 0:
            raise ValueError("旧K/V必须是非空四维张量")
        if old.shape[:2] != new.shape[:2] or old.shape[3] != new.shape[3]:
            raise ValueError("旧/新B,H,d不一致")
        if old.dtype != new.dtype or old.device != new.device:
            raise ValueError("旧/新dtype或device不一致")
    if past[0].shape != past[1].shape:
        raise ValueError("旧K/V形状不匹配")
    return torch.cat((past[0], k_new), dim=-2), torch.cat((past[1], v_new), dim=-2)

check_exercise_b(append_kv_answer)

练习 B 通过


### C：逐块前向

注意 `start` 每次累计 `length`，不是一直取 `ids[:, :length]`。

In [28]:
def incremental_logits_answer(model, ids, chunks):
    if (not isinstance(chunks, list) or not chunks
            or any(type(n) is not int or n <= 0 for n in chunks)
            or sum(chunks) != ids.shape[1]):
        raise ValueError("chunk必须是正整数，且总和等于序列长度")
    was_training = model.training
    cache, results, start = None, [], 0
    try:
        model.eval()
        with torch.no_grad():
            for length in chunks:
                new_ids = ids[:, start:start + length]
                logits, cache = model.forward_cached(new_ids, cache)
                results.append(logits)
                start += length
            return torch.cat(results, dim=1)
    finally:
        model.train(was_training)

check_exercise_c(incremental_logits_answer)

练习 C 通过


## 13. 验收、排错与下一课

你应能解释下面六件事：

1. 为什么 P 个提示、N 个输出只需 N-1 次额外 decode？
2. 为什么每层要保存自己的 K/V，而不是只存最后一层或旧 Q？
3. `past_len=3,new_len=2` 的 mask 为什么不是左上角普通三角？
4. 为什么原来的 cache 可以分叉，但改了早期 token 要重建？
5. 缓存长度翻倍时，载荷和单步 attention 工作量怎样变化？
6. 为什么 logits 近似一致、生成文字一致和运行更快是三个不同的检查？

| 现象 | 先检查 |
|---|---|
| Prefill 对，但第二步开始不一致 | 位置是否仍从0开始；是否又传了整个旧前缀 |
| 单token对，chunk不对 | 偏移mask、K/V顺序、chunk内部因果性 |
| cache的长度增长异常 | `cat` 是否沿时间轴；是否重复缓存旧token |
| 内存越用越大 | 是否保存所有历史cache、仍在建计算图、上下文越来越长 |
| 加速比很小或变慢 | 模型/前缀大小、Python校验和cat开销、CUDA同步、是否公平比较 |

独立脚本命令：

```bash
python lesson_19_kv_cache_prefill_and_decode.py
python lesson_19_kv_cache_prefill_and_decode.py --checks-only
python lesson_19_kv_cache_prefill_and_decode.py --checkpoint lesson_17_outputs/best_model.pt
python lesson_19_kv_cache_prefill_and_decode.py --checkpoint lesson_18_outputs/frozen_model.pt --device auto
python lesson_19_kv_cache_prefill_and_decode.py --skip-benchmark
```

下一课：**RoPE 旋转位置编码**。从本课的可学习绝对位置出发，研究旋转如何作用在 Q/K 上，
以及有 cache 时怎样保证旋转位置正确。不能把已有绝对位置模型直接切换成 RoPE 就假定权重仍有效。

## 资料与实现范围

本课公式、小例子、测试和测量由本讲义独立推导或运行。API 行为核对以下官方资料：

1. [Hugging Face — How caching works](https://huggingface.co/docs/transformers/cache_explanation)：逐层缓存、历史与新增 token。
2. [Hugging Face — Cache strategies](https://huggingface.co/docs/transformers/kv_cache)：动态、静态与量化等缓存的边界。
3. [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)：布尔 mask、非方阵因果对齐和 Dropout。
4. [CUDA synchronize](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html) 与 [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html)：异步执行和同步计时。
5. [PyTorch numerical accuracy](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)：浮点运算顺序与误差。

默认两层教学模型，不是主流大模型的部署性能基准。没有验证 RTX 3060；`--device auto` 提供运行入口，
实际显存、速度和误差需要在你的设备上测量。